In [1]:
import os

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

d:\miniconda3\envs\PyTorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import pandas as pd

data = pd.read_csv("./ChnSentiCorp_htl_all.csv")
data

,label,review
0,1,"距离川沙公路较近,但是公交指示不对,如果是""蔡陆线""的话,会非常麻烦.建议用别的路线.房间较..."
1,1,商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!
2,1,早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。
3,1,宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小...
4,1,"CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风"
...,...,...
7761,0,尼斯酒店的几大特点：噪音大、环境差、配置低、服务效率低。如：1、隔壁歌厅的声音闹至午夜3点许...
7762,0,盐城来了很多次，第一次住盐阜宾馆，我的确很失望整个墙壁黑咕隆咚的，好像被烟熏过一样家具非常的...
7763,0,看照片觉得还挺不错的，又是4星级的，但入住以后除了后悔没有别的，房间挺大但空空的，早餐是有但...
7764,0,我们去盐城的时候那里的最低气温只有4度，晚上冷得要死，居然还不开空调，投诉到酒店客房部，得到...


In [4]:
data = data.dropna()
data

,label,review
0,1,"距离川沙公路较近,但是公交指示不对,如果是""蔡陆线""的话,会非常麻烦.建议用别的路线.房间较..."
1,1,商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!
2,1,早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。
3,1,宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小...
4,1,"CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风"
...,...,...
7761,0,尼斯酒店的几大特点：噪音大、环境差、配置低、服务效率低。如：1、隔壁歌厅的声音闹至午夜3点许...
7762,0,盐城来了很多次，第一次住盐阜宾馆，我的确很失望整个墙壁黑咕隆咚的，好像被烟熏过一样家具非常的...
7763,0,看照片觉得还挺不错的，又是4星级的，但入住以后除了后悔没有别的，房间挺大但空空的，早餐是有但...
7764,0,我们去盐城的时候那里的最低气温只有4度，晚上冷得要死，居然还不开空调，投诉到酒店客房部，得到...


In [5]:
data.iloc[0].label, data.iloc[0].review

(np.int64(1), '距离川沙公路较近,但是公交指示不对,如果是"蔡陆线"的话,会非常麻烦.建议用别的路线.房间较为简单.')

In [6]:
from torch.utils.data import Dataset

class MyDataset(Dataset):

    def __init__(self) -> None:
        super().__init__()
        self.data = pd.read_csv("./ChnSentiCorp_htl_all.csv")
        self.data = self.data.dropna()

    def __getitem__(self, index):
        return self.data.iloc[index]["review"], self.data.iloc[index]["label"]
    
    def __len__(self):
        return len(self.data)

In [7]:
dataset = MyDataset()
for i in range(5):
    print(dataset[i])
review, label = dataset[0]
label, review

('距离川沙公路较近,但是公交指示不对,如果是"蔡陆线"的话,会非常麻烦.建议用别的路线.房间较为简单.', np.int64(1))
('商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!', np.int64(1))
('早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。', np.int64(1))
('宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小，但加上低价位因素，还是无超所值的；环境不错，就在小胡同内，安静整洁，暖气好足-_-||。。。呵还有一大优势就是从宾馆出发，步行不到十分钟就可以到梅兰芳故居等等，京味小胡同，北海距离好近呢。总之，不错。推荐给节约消费的自助游朋友~比较划算，附近特色小吃很多~', np.int64(1))
('CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风', np.int64(1))


(np.int64(1), '距离川沙公路较近,但是公交指示不对,如果是"蔡陆线"的话,会非常麻烦.建议用别的路线.房间较为简单.')

In [8]:
from torch.utils.data import random_split

trainset, validset = random_split(dataset, lengths=[0.9, 0.1])
len(trainset), len(validset)

(6989, 776)

In [9]:
for i in range(10):
    print(trainset[i])

('我是在网上预订中煤的2007年8月22日-8月25日在宾馆住了3天。感觉在旅游旺季在秦皇岛3星级的宾馆每天240的价格是非常物有所值的，每天早起每人15元的自助餐花样较多吃的相当不错，干净卫生，住宿环境非常好，虽然离相关的旅游景点较远，但交通还是便利的出门做9路车4站地到火车站，而且车的班次非常快1～2分钟就能等上车，价格很有优势。去其他的景点做9路到四道桥到车也非常方便，前台的服务人员热情。', np.int64(1))
('酒店的外、内部环境还是很不错的，服务生的服务意识也还可以，但里面吃的东东有些贵，另外携程确认定单时间有些拖延，打车到了宾馆后，被通知没有３２０元的房间了，只能住５２０的了，呜．．．对于自助旅行的偶来讲，有些奢侈．．．．．', np.int64(1))
('从市内打车要15元，偏远了一点，不过到了以后才发现环境不错，比较满意', np.int64(1))
('交通很方便，从罗湖火车站直接可以坐地铁去，在世界之窗站下一出站就是了。房间很干净，阳台也比较大。最喜欢就是里面的游泳池，很有威尼斯的特色，小朋友去就很喜欢了。checkin和checkout的时候有很新鲜的苹果吃，呵呵，感觉很舒服！', np.int64(1))
('酒店的checkin和checkout很快.预订的普通单人间没有了免费升到了商务单人单,不错,房间内的电脑网速也是很快.就是地毯老了点.可以就是A幢和B幢的区别吧.早餐也还可以吧.总体来讲性价比挺高.下次还会选择.', np.int64(1))
('五一住了这里，作为三星级硬件还可以。前台经理赞一个，我们一行在隔壁吃饭，发票中了奖，但饭馆就是不肯帮我们兑，态度也很无理。我们将此事说与宾馆，前台张颖经理说帮我们处理，并为我们邮回北京，虽然钱还没收到，但是我们心里很愉快，毕竟这种专业的责任意识与服务意识，在中华大地是值得提倡的。', np.int64(1))
('房间很好，设施也很新，总的来说还是很满意的。周围的很方便，楼下隔壁都有商场。', np.int64(1))
('该酒店地段不方便，超难打车，服务不到位，酒店设备有点旧', np.int64(0))
('上个周末又去了河源。这回提前预订了万豪国际酒店2个晚上，酒店真的没有令我们夫妇失望，我们为自己的选择洋洋得意。当时由于不能在约定的时间傍晚6点钟之前到达酒店，和酒店联系

In [10]:
import torch

tokenizer = AutoTokenizer.from_pretrained("hfl/rbt3")

def collate_func(batch):
    texts, labels = [], []
    for item in batch:
        texts.append(item[0])
        labels.append(item[1])
    inputs = tokenizer(texts, max_length=128, padding="max_length", truncation=True, return_tensors="pt")
    inputs["labels"] = torch.tensor(labels)
    return inputs

In [11]:
from torch.utils.data import DataLoader

trainloader = DataLoader(trainset, batch_size=32, shuffle=True, collate_fn=collate_func)
validloader = DataLoader(validset, batch_size=64, shuffle=False, collate_fn=collate_func)

In [12]:
next(enumerate(trainloader))

(0,
 {'input_ids': tensor([[ 101, 7478, 2382,  ...,    0,    0,    0],
         [ 101, 1765, 4415,  ...,    0,    0,    0],
         [ 101,  769, 6858,  ...,    0,    0,    0],
         ...,
         [ 101, 4696, 4638,  ..., 4507, 3204,  102],
         [ 101,  679, 7231,  ...,    0,    0,    0],
         [ 101, 6983, 2421,  ...,    0,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         ...,
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         ...,
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0]]), 'labels': tensor([1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1])})

In [13]:
from torch.optim import Adam

model = AutoModelForSequenceClassification.from_pretrained("hfl/rbt3")

if torch.cuda.is_available():
    model = model.cuda()
model

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at hfl/rbt3 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(21128, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-2): 3 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-1

In [14]:
optimizer = Adam(model.parameters(), lr=2e-5)

In [15]:
for item in trainloader:
   for k, v in item.items():
       print(f"key: {k}, value: {v}")
   break

key: input_ids, value: tensor([[ 101,  679, 7231,  ...,    0,    0,    0],
        [ 101, 2697, 6230,  ...,    0,    0,    0],
        [ 101,  122,  119,  ..., 1168, 1373,  102],
        ...,
        [ 101, 4895, 1062,  ...,    0,    0,    0],
        [ 101, 6983, 2421,  ...,    0,    0,    0],
        [ 101, 8037, 9414,  ...,    0,    0,    0]])
key: token_type_ids, value: tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]])
key: attention_mask, value: tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])
key: labels, value: tensor([1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1,
        0, 1, 1, 1, 1, 1, 1, 1])


In [16]:
def evaluate():
    model.eval()
    acc_num = 0
    with torch.inference_mode():
        for batch in validloader:
            if torch.cuda.is_available():
                batch = {k: v.cuda() for k, v in batch.items()}
            output = model(**batch)
            pred = torch.argmax(output.logits, dim=-1)
            acc_num += (pred.long() == batch["labels"].long()).float().sum()
    return acc_num / len(validset)

def train(epoch=5, log_step=100):
    global_step = 0
    for ep in range(epoch):
        model.train()
        for batch in trainloader:
            if torch.cuda.is_available():
                batch = {k: v.cuda() for k, v in batch.items()}
            optimizer.zero_grad()
            output = model(**batch)
            output.loss.backward()
            optimizer.step()
            if global_step % log_step == 0:
                print(f"ep: {ep}, global_step: {global_step}, loss: {output.loss.item()}")
            global_step += 1
        acc = evaluate()
        print(f"ep: {ep}, acc: {acc}")

In [17]:
train()

ep: 0, global_step: 0, loss: 0.6278400421142578
ep: 0, global_step: 100, loss: 0.1536608636379242
ep: 0, global_step: 200, loss: 0.11604151874780655
ep: 0, acc: 0.8827319145202637
ep: 1, global_step: 300, loss: 0.2810283899307251
ep: 1, global_step: 400, loss: 0.22226548194885254
ep: 1, acc: 0.8878865838050842
ep: 2, global_step: 500, loss: 0.1480904221534729
ep: 2, global_step: 600, loss: 0.33176565170288086
ep: 2, acc: 0.8827319145202637
ep: 3, global_step: 700, loss: 0.22531266510486603
ep: 3, global_step: 800, loss: 0.11049988865852356
ep: 3, acc: 0.8878865838050842
ep: 4, global_step: 900, loss: 0.10507895797491074
ep: 4, global_step: 1000, loss: 0.10824534296989441
ep: 4, acc: 0.8969072103500366


In [18]:
sen = "我觉得这家酒店不错，饭很好吃！"
id2_label = {0: "差评！", 1: "好评！"}
model.eval()
with torch.inference_mode():
    inputs = tokenizer(sen, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    logits = model(**inputs).logits
    pred = torch.argmax(logits, dim=-1)
    print(f"输入：{sen}\n模型预测结果:{id2_label.get(pred.item())}")

输入：我觉得这家酒店不错，饭很好吃！
模型预测结果:好评！


In [19]:
from transformers import pipeline

model.config.id2label = id2_label
pipe = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)

Device set to use cuda:0


In [20]:
pipe(sen)

[{'label': '好评！', 'score': 0.9713993072509766}]